In [22]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, r2_score

In [23]:
# Upload the dataset file into Google Colab
df = pd.read_csv("../dataset/MumbaiPriceCropDataset.csv")
df.head()

,State,District,Market,Commodity,Variety,Grade,Arrival_Date,Min_Price,Max_Price,Modal_Price,Commodity_Code
0,Maharashtra,Mumbai,Mumbai,Ridgeguard(Tori),Other,FAQ,05-01-2015,4000,4400,4200,160
1,Maharashtra,Mumbai,Mumbai,Ridgeguard(Tori),Other,FAQ,07-01-2015,3800,4200,4000,160
2,Maharashtra,Mumbai,Mumbai,Ridgeguard(Tori),Other,FAQ,26-01-2015,3200,3400,3300,160
3,Maharashtra,Mumbai,Mumbai,Ridgeguard(Tori),Other,FAQ,28-01-2015,2800,3000,2900,160
4,Maharashtra,Mumbai,Mumbai,Ridgeguard(Tori),Other,FAQ,03-02-2015,2000,2400,2200,160


In [24]:
df["Arrival_Date"] = pd.to_datetime(df["Arrival_Date"], dayfirst=True)

In [25]:
# Convert Arrival_Date to datetime
df["Arrival_Date"] = pd.to_datetime(df["Arrival_Date"], dayfirst=True)

# Extract useful time features
df["Year"] = df["Arrival_Date"].dt.year
df["Month"] = df["Arrival_Date"].dt.month
df["Day"] = df["Arrival_Date"].dt.day

In [26]:
# Remove columns that cause data leakage or are not useful
df = df.drop([
    "Arrival_Date",
    "Min_Price",
    "Max_Price",
    "Commodity_Code"
], axis=1)

In [27]:
# Encode categorical text columns into numerical values using LabelEncoder
le_state = LabelEncoder()
le_district = LabelEncoder()
le_market = LabelEncoder()
le_commodity = LabelEncoder()
le_variety = LabelEncoder()
le_grade = LabelEncoder()

df["State"] = le_state.fit_transform(df["State"])
df["District"] = le_district.fit_transform(df["District"])
df["Market"] = le_market.fit_transform(df["Market"])
df["Commodity"] = le_commodity.fit_transform(df["Commodity"])
df["Variety"] = le_variety.fit_transform(df["Variety"])
df["Grade"] = le_grade.fit_transform(df["Grade"])

In [28]:
# Separate input features (X) and target variable (y)
X = df.drop(["Modal_Price"], axis=1)

y = df["Modal_Price"]

In [29]:
# Split the dataset into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [30]:
# Train the Random Forest regression model using training data
model = RandomForestRegressor()

model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [31]:
# Predict crop prices using the trained model
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print("MAE:", mae)
print("R2 Score:", r2)

MAE: 464.0116066666667
R2 Score: 0.9828189724222105


In [32]:
state = le_state.transform(["Maharashtra"])[0]
district = le_district.transform(["Mumbai"])[0]
market = le_market.transform(["Mumbai"])[0]
commodity = le_commodity.transform(["Rice"])[0]
variety = le_variety.transform(["Other"])[0]
grade = le_grade.transform(["FAQ"])[0]

year = 2026
month = 7
day = 15

prediction = model.predict([[state,district,market,commodity,variety,grade,year,month,day]])

print("Predicted Price:", prediction[0])

Predicted Price: 4085.5


c:\Users\amogh kotawadekar\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [33]:
input_data = pd.DataFrame({
    "State":[state],
    "District":[district],
    "Market":[market],
    "Commodity":[commodity],
    "Variety":[variety],
    "Grade":[grade],
    "Year":[year],
    "Month":[month],
    "Day":[day]
})

prediction = model.predict(input_data)

print("Predicted Price:", prediction[0])

Predicted Price: 4085.5


In [34]:
print(X.columns)

Index(['State', 'District', 'Market', 'Commodity', 'Variety', 'Grade', 'Year',
       'Month', 'Day'],
      dtype='object')


In [35]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor


models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),
    "KNN": KNeighborsRegressor(),

}

for name, model in models.items():

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    print(name)
    print("MAE:", mae)
    print("R2 Score:", r2)
    print("-----------------------")

Linear Regression
MAE: 7190.777487608554
R2 Score: 0.03863518765954943
-----------------------
Decision Tree
MAE: 476.3445
R2 Score: 0.9745679825266065
-----------------------
Random Forest
MAE: 461.47375
R2 Score: 0.9826311051888061
-----------------------
KNN
MAE: 4755.130433333334
R2 Score: 0.43700317954688483
-----------------------


The best model should have:

Highest R²

Lowest MAE

MAE: Mean Absolute Error
Average difference between actual price and predicted price.

R² Score — Coefficient of Determination

R² = 1 → perfect model
R² = 0 → model useless
R² < 0 → worse than random guessing

The Random Forest Regressor performed the best.

Reasons:

Highest R² Score: 0.982

Lowest MAE: 467

This means the model explains 98.2% of the variation in crop prices and the average prediction error is about ₹467.

In [38]:
# Train the best performing model (Random Forest)
best_model = RandomForestRegressor()
best_model.fit(X_train, y_train)

# Save the trained model to a .pkl file for future predictions
import pickle
pickle.dump(best_model, open("../../ML_models/price_model.pkl","wb"))

In [39]:
import pickle

# Load trained crop price prediction model
loaded_model = pickle.load(open("../../ML_models/price_model.pkl", "rb"))

print("Model loaded successfully!")

Model loaded successfully!


In [ ]:
#checking if model saved or nott
sample = X_test.iloc[0:1]
prediction = loaded_model.predict(sample)

print("Predicted Price:", prediction)

Predicted Price: [9129.]


In [41]:
sample = X_test.iloc[0:1]

prediction = loaded_model.predict(sample)

actual = y_test.iloc[0]

print("Predicted Price:", prediction[0])
print("Actual Price:", actual)

Predicted Price: 9129.0
Actual Price: 9100
